In [1]:
import tensorflow as tf

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

class_names = train_dataset.class_names
print("class_names:", class_names) 

Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.
class_names: ['cat', 'dog']


In [6]:
def flip_left_right(image, label):
    return tf.image.flip_left_right(image), label

def flip_up_down(image, label):
    return tf.image.flip_up_down(image), label

def rot90(image, label):
    return tf.image.rot90(image), label

def rot180(image, label):
    return tf.image.rot90(image, k=2), label

def rot270(image, label):
    return tf.image.rot90(image, k=3), label


train_lr     = train_dataset.map(flip_left_right)
train_ud     = train_dataset.map(flip_up_down)
train_rot90  = train_dataset.map(rot90)
train_rot180 = train_dataset.map(rot180)
train_rot270 = train_dataset.map(rot270)


train_dataset = (train_dataset
                 .concatenate(train_lr)
                 .concatenate(train_ud)
                 .concatenate(train_rot90)
                 .concatenate(train_rot180)
                 .concatenate(train_rot270))

train_dataset = train_dataset.shuffle(32)

In [7]:
# MobileNetV2モデルを作成する
input_layer = tf.keras.Input(shape=(224, 224, 3))   # 入力層
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)   # 前処理（正規化）をする層

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(224, 224, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False

# Dense層を追加する
output_layer = tf.keras.layers.Dense(1, activation='sigmoid')

# base_modelに先ほどのDense層を追加したモデルを作成する
model = tf.keras.Sequential([
    base_model,
    output_layer
])

# modelをcompileする
model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])

In [8]:
# modelに学習させる
model.fit(train_dataset, epochs=20)

Epoch 1/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 832s 384ms/step - accuracy: 0.9829 - loss: 0.0655
Epoch 2/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 824s 381ms/step - accuracy: 1.0000 - loss: 0.0085
Epoch 3/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 2515s 1s/step - accuracy: 1.0000 - loss: 0.0027
Epoch 4/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 567s 262ms/step - accuracy: 1.0000 - loss: 9.8930e-04
Epoch 5/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 633s 292ms/step - accuracy: 1.0000 - loss: 3.7925e-04
Epoch 6/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 1601s 741ms/step - accuracy: 1.0000 - loss: 1.4720e-04
Epoch 7/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 951s 440ms/step - accuracy: 1.0000 - loss: 5.7397e-05
Epoch 8/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 2497s 1s/step - accuracy: 1.0000 - loss: 2.2342e-05
Epoch 9/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 3129s 1s/step - accuracy: 1.0000 - loss: 8.7321e-06
Epoch 10/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 2359s 1s/step - accuracy: 1.0000 - loss: 3.4657e-06
Epoch 11/20
2160/2160 ━━━━━━━━━━━━━━━━━━━━ 658s 304ms/step

In [9]:
pred_data = model.predict(test_dataset)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 284ms/step


In [10]:
pred_data

array([[6.2643278e-26],
       [2.5566028e-21],
       [1.2553904e-24],
       [1.1379988e-17],
       [7.5290537e-21],
       [2.6738242e-21],
       [4.7474993e-21],
       [4.2866114e-17],
       [1.9393049e-32],
       [3.4857077e-18],
       [7.8161528e-26],
       [4.2156851e-15],
       [1.3746963e-13],
       [2.9705869e-17],
       [9.2069426e-25],
       [7.3601529e-33],
       [2.8727518e-24],
       [2.3447675e-19],
       [2.2345259e-12],
       [4.7835549e-14],
       [3.9536662e-11],
       [2.9107088e-09],
       [2.3051758e-12],
       [1.3137758e-19],
       [1.9124602e-06],
       [1.2970560e-16],
       [7.6205662e-20],
       [2.2152131e-16],
       [5.5032595e-13],
       [1.5057546e-22],
       [1.0141510e-18],
       [1.7067653e-01],
       [4.0770967e-20],
       [3.8399040e-16],
       [1.2030129e-12],
       [1.0000000e+00],
       [2.9820599e-13],
       [8.3127714e-17],
       [2.4240164e-06],
       [4.1142692e-19],
       [1.0698256e-18],
       [5.835208

In [11]:
# evaluate()でモデルの性能を評価する
model.evaluate(test_dataset)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 173ms/step - accuracy: 0.9700 - loss: 0.3297


[0.3296666443347931, 0.9700000286102295]